In [ ]:
# PyTorch functions/methods helpers

# 6.3.4
print("xavier std:", float(xavier_layer.weight.std())) # Xavier std ≈ 2 / (128 + 64) ≈ 1 / 96
print("kaiming std:", float(kaiming_layer.weight.std())) # Kaiming std ≈ 2 / 128 ≈ 1 / 64

# 6.3.5
net.apply(init_mlp) # Run this function on every module inside this network (instead of applying individually at net[0], net[1]...)

for name, param in net.named_parameters():
    print(name, shape(param), float(param.std()) if param.ndim > 1 else float(param.sum())) # Print name, shape and std for weights (where ndim > 1) or sum for bias (where ndim = 1 or vectors)

* Training does not begin from nowhere.

* Before the first batch, every learned tensor already has values, and those starting values shape the early forward signals and backward gradients.

* Initialization is the bridge between architecture design and optimization behavior.

# How to use this notebook

* Run the notebook from top to bottom.

* Every code block is designed to be cloud-runnable and self-contained inside this notebook.

* The drills are intentionally small: predict the shape or behavior first, run the cell, then read the assertion as the contract you must understand.

# You are done when you can

- explain why starting values affect optimization even before learning begins
- inspect initialized weights and biases
- explain why all-zero weights create symmetry problems
- connect activation scale to gradient usability
- compare tiny, huge, Xavier, and Kaiming-style initializations
- apply an initializer across a module tree

In [1]:
import math
from pathlib import Path
import tempfile

import torch
from torch import nn

torch.manual_seed(0)
torch.set_printoptions(precision=4, sci_mode=False)

def shape(x):
    return tuple(x.shape)

def count_scalars(parameters):
    return sum(p.numel() for p in parameters)

# 6.3.0 The Problem This Notebook Solves

A model architecture says what computations are possible.

Initialization decides where optimization starts inside that space of possible computations.

For a linear layer, the weights and biases are trainable, but before training they are just initial guesses.

Bad guesses can make learning unnecessarily hard:

- If weights are too small, **signals can shrink** as they pass through many layers.
- If weights are too large, **activations and gradients can explode**.
- If many units start exactly the same, they can receive the same gradients and learn redundant features.

The goal is not to find magical starting values.

The goal is to start in a regime where different units can learn different things and gradients remain numerically useful.

This connects back to Chapter 5's stability discussion.
* There, instability was about forward and backward propagation through deep networks.
* Here, we look at the software mechanism that chooses the initial tensor values.

# 6.3.1 Inspect Before You Trust

Before reasoning about initialization, look at the actual tensor statistics.

The exact random values do not matter. **The scale, shape, and finiteness matter**.
* The weight shape of `nn.Linear(8, 4)` is `(4, 8)` because the layer produces 4 output features from 8 input features.
* Each output feature owns one row of 8 incoming weights.
* The bias shape is `(4,)` because each output feature gets one bias.

The mean and standard deviation are quick sanity checks:

- mean near zero usually means the layer does not start with a strong systematic positive or negative preference
- standard deviation tells you the typical weight size
- finite values rule out broken initialization

This is the first habit: inspect before trusting an initializer.

In [ ]:
layer = nn.Linear(8, 4)

print("weight shape:", shape(layer.weight))
print("bias shape:", shape(layer.bias))
print("bias shape:", float(layer.weight.mean()))
print("bias shape:", shape(layer.weight.std()))

assert shape(layer.weight) == (4, 8)
assert shape(layer.bias) == (4, )
assert torch.isfinite(layer.weight).all()

weight shape: (4, 8)
bias shape: (4,)
bias shape: -0.017209907993674278
bias shape: ()


/tmp/ipykernel_1788/2811092179.py:5: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print("bias shape:", float(layer.weight.mean()))


# 6.3.2 Zero Weights Are Mechanically Bad for Hidden Units

All-zero weights are not just "small." They create symmetry.
* If several hidden units start with the same incoming weights and the same bias, they compute the same output for the same input.
* If they also receive the same gradient pattern, they can keep moving together.
* **The model wastes capacity because multiple units behave like copies of each other**.

Biases are different.
* A zero bias does not usually force two units to have identical incoming feature detectors if their weights are different.
* **Even if two neurons have a bias of 0, they can still learn completely different things as far as their weights are different**.
* That is why zero bias initialization is common, while all-zero weight initialization is usually a bad default for hidden layers.

In this tiny example, every output unit produces the exact same value: zero.

The point is not that zero output is always bad; the point is that the hidden units are indistinguishable.

In [ ]:
layer = nn.Linear(3, 4)

with torch.no_grad():
    nn.init.zeros_(layer.weight) # shape of (4, 3)
    nn.init.zeros_(layer.bias) # shape of (4, )

X = torch.tensor([1.0, 2.0, 3.0]) # shape of (3, )
Y = layer(X) # (3, ) @ (3, 4) = (4, ) since X is treated as a vector and multiply directly against weight.T to get another vector

print(Y)

# assert torch.equal(Y, torch.zeros(1, 4)) # Incorrect, should be torch.zeros(4, )
assert torch.equal(Y, torch.zeros(4))
# assert torch.equal(Y[:0], Y[:, 1]) # Incorrect, since Y only has 1 dimension (is a vector)
assert torch.equal(Y[0], Y[1])

tensor([0., 0., 0., 0.], grad_fn=<ViewBackward0>)


# 6.3.3 Initialization Scale Changes Activation Scale

Deep networks repeatedly transform representations.

Even before training, each layer changes the scale of the tensor it receives.

If each layer shrinks the representation too much, later layers receive almost no signal.

If each layer amplifies the representation too much, values can become huge. In both cases, gradients can become hard to use.

This cell does not train. That is intentional. It isolates one question:

```text
what happens to activation scale during a forward pass through several randomly initialized layers?
```

* The tiny initialization should tend to shrink activations across layers.
* The huge initialization should tend to expand them.
* This gives a concrete reason fan-aware initializers exist.

In [ ]:
def activation_std_after_stack(init_std):

    torch.manual_seed(0)
    layers = []

    for _ in range(5):

        # Initialize layer, weights and bias
        layer = nn.Linear(64, 64)
        with torch.no_grad():
            nn.init.normal_(layer.weight, mean=0.0, std=init_std) # For every layer, initialize its weights using this same standard deviation.
            nn.init.zeros_(layer.bias)
        layers += [layer, nn.ReLU()] # Layer is iterated with ReLU, so layer 1 -> ReLU -> layer 2 -> ReLU -> ...

    X = torch.randn(256, 64)
    stats = []

    for layer in layers: # Run X, shape of (256, 64), through all layers
        X = layer(X)
        if isinstance(layer, nn.Linear): # If X goes through a linear layer, record its standard deviation
            stats.append(float(X.std()))
    return stats

tiny_stats = activation_std_after_stack(0.01)
huge_stats = activation_std_after_stack(1.0)

print("tiny init stds:", tiny_stats)
print("huge init stds:", huge_stats)

assert tiny_stats[-1] < tiny_stats[0] # Activations/layer outputs get scaled DOWN at every layer since every layer has SMALL weights (weight values concentrate near mean=0)
assert huge_stats[-1] > huge_stats[0] # Activations/layer outputs get scaled UP at every layer since every layer has HUGE weights (weight values spread further from mean=0)

tiny init stds: [0.07975491136312485, 0.004536441992968321, 0.00024252179719042033, 1.439409788872581e-05, 7.984618264345045e-07]
huge init stds: [7.975491046905518, 45.36442184448242, 242.5218048095703, 1439.409912109375, 7984.61962890625]


# 6.3.4 Use Fan-Aware Initializers for Common Layers

Fan-aware initializers choose weight scale using the layer's width.

Plain-English idea:

- A layer with many inputs adds together many weighted values.
- If the weights are not scaled carefully, the sum can become too large or too small.
- The right scale depends on how many values flow into or out of the layer.

Xavier initialization is commonly associated with activations that are roughly **symmetric around zero**.
* "Keep signal stable assuming the activation roughly preserves both sides."

Kaiming initialization is commonly used with **ReLU-like activations** because ReLU discards negative values and changes signal flow.
* "Since my weigths will pass through ReLU later, initialization will scale to account for ReLU (kill negatives)."

Initialization is important because training can't necessarily recover from bad starting weights.

If activations/gradients vanish as they pass through the layers because the weights are too small, or explode because the weights are too large, training can become ineffective (signal vanish/explode).

You do not need to memorize the derivation here.

The rigor is knowing what problem these initializers solve: **preserve usable signal scale through the network at the start of training**.

In [ ]:
xavier_layer = nn.Linear(128, 64)
kaiming_layer = nn.Linear(128, 64)

with torch.no_grad():
    # Initiaze Xavier weights and bias
    nn.init.xavier_uniform_(xavier_layer.weight)
    nn.init.zeros_(xavier_layer.bias)

    # Initiaze Kaiming weights and bias
    nn.init.kaiming_uniform_(kaiming_layer.weight, nonlinearity="relu")
    nn.init.zeros_(kaiming_layer.bias)

print("xavier std:", float(xavier_layer.weight.std())) # Xavier std ≈ 2 / (128 + 64) ≈ 1 / 96
print("kaiming std:", float(kaiming_layer.weight.std())) # Kaiming std ≈ 2 / 128 ≈ 1 / 64

assert torch.isfinite(xavier_layer.weight).all()
assert torch.isfinite(kaiming_layer.weight).all()

xavier std: 0.10223204642534256
kaiming std: 0.1246219128370285


# 6.3.5 Apply Initialization Across a Module Tree

Real models contain many modules.
* You do not want to manually initialize each layer one at a time.
* You definitely do not want to accidentally initialize a `ReLU` as if it had weights.
* `net.apply(fn)` walks through the module tree.
* The initializer receives each module, checks what type it is, and acts only when appropriate.

This is where Chapters 6.1 and 6.2 connect:

- 6.1 gave you the module tree.
- 6.2 gave you parameter ownership.
- 6.3 now uses the module tree to mutate parameters deliberately.

The type check is not optional ceremony.

It is the guardrail that **keeps model-wide initialization from touching objects that do not own the expected parameter shapes**.

In [ ]:
def init_mlp(module):

    # Initialize params with Kaiming
    if isinstance(module, nn.Linear): # Only initialize the params at linear layers (r.e. non ReLU layers)
        nn.init.kaiming_uniform_(module.weight, nonlinearity="relu")
        nn.init.zeros_(module.bias)

net = nn.Sequential(
    nn.Linear(10, 20),
    nn.ReLU(),
    nn.Linear(20, 5),
)

net.apply(init_mlp) # Run this function on every module inside this network (instead of applying individually at net[0], net[1]...)

for name, param in net.named_parameters():
    print(name, shape(param), float(param.std()) if param.ndim > 1 else float(param.sum())) # Print name, shape and std for weights (where ndim > 1) or sum for bias (where ndim = 1 or vectors)

assert torch.equal(net[0].bias, torch.zeros_like(net[0].bias))
assert torch.equal(net[2].bias, torch.zeros_like(net[2].bias))

# 6.3.6 Break It Deliberately: Initialize Too Late

Initialization is supposed to happen before learning.

Reinitialization after training is not a harmless refresh; it overwrites the learned function.

The theory-level mistake is confusing setup with training.
* During training, the optimizer gradually changes parameters to reduce loss.
* If you call an initializer afterward, you replace those learned values with a new starting point.
* The model may still have the same architecture, but its learned behavior is gone.

This cell demonstrates the mutation directly by saving a copy of the weight, overwriting it, and proving the value changed.

In [ ]:
layer = nn.Linear(2, 1) # Weight has shape (1, 2) and is automatically initialized when nn.Linear() creates the layer (via uniform random initialization)
before = layer.weight.detach().clone()

with torch.no_grad():
    nn.init.ones_(layer.weight)

after = layer.weight.detach().clone()

print("before:", before)
print("after:", after) # All values are now 1, but the shape remains unchanged from before

assert not torch.allclose(before, after)
assert torch.equal(after, torch.ones_like(after))

before: tensor([[-0.0407,  0.7012]])
after: tensor([[1., 1.]])


# 6.3 Checkpoint

Answer these before moving on.

You do not need a separate notes file for chapters; short answers in markdown cells or in your own study notes are enough.

1. Why is initialization part of optimization theory, not just software setup?
> Because bad weights actually impact training performance (too small cause signal vanishing and too large cause signal explosion)

2. Why are zero biases usually less dangerous than zero weights?
> Even if two neurons have a bias of 0, they can still learn completely different things as far as their weights are different

3. What symmetry problem do all-zero hidden weights create?
> If all hidden weights are initialized to the same value (especially zero), the neurons produce the same outputs and receive the same gradients. Therefore, they remain identical and cannot learn different features

4. What did the tiny and huge initialization drill show mechanically?
> Too small weights cause signal vanishing and too large weights cause signal explosion

5. What problem are Xavier and Kaiming initializers trying to solve?
> Xavier accounts for the input and output dimensions, while Kaiming additionally accounts for the effect of ReLU-like activations

6. Why should an initializer check `isinstance(module, nn.Linear)` before touching `.weight`?
> Different module types have different parameters/structures (e.g. ReLU, BatchNorm, LayerNorm), so you shouldn't assume every module has a `.weight` that should be initialized this way

7. Why is reinitializing after training equivalent to discarding learned parameters?
> Because the weights' values already changed during training. Initialization literally means do before training/pre-training